# Population and Sample Variance

*Explain the n − 1 correction and distinguish describing observed values from estimating a population variance.*


## Two Denominators Answer Different Descriptive Tasks

For a complete finite population, divide squared deviations from its mean by
$N$. For an independent, identically distributed random sample of size $n>1$
from a population with finite variance, the conventional unbiased estimator is

$$s^2=\frac{\sum_{i=1}^{n}(x_i-\bar{x})^2}{n-1}.$$

Using $n$ still describes the spread of the observed sample itself. The
$n-1$ correction addresses estimation bias when the unknown population mean
has been replaced by the sample mean. The choice follows the purpose of the
calculation, not a rule that every small table must use one denominator.


## Estimating the Mean Uses One Degree of Freedom

Deviations from the sample mean must sum to zero. Once the first `n - 1`
deviations are known, the last is determined. `ddof` means that the divisor
is the count minus `ddof`; specify it explicitly when comparing libraries.


In [1]:
import numpy as np
import pandas as pd

sample = np.array([1, 3, 5, 7, 9], dtype=float)
deviations = sample - sample.mean()
print("First four deviations:", deviations[:-1])
print("Required last deviation:", -deviations[:-1].sum())
print("Variance with divisor n:", np.var(sample, ddof=0))
print("Sample variance with divisor n - 1:", np.var(sample, ddof=1))
print("pandas sample variance:", pd.Series(sample).var(ddof=1))


First four deviations: [-4. -2.  0.  2.]
Required last deviation: 4.0
Variance with divisor n: 8.0
Sample variance with divisor n - 1: 10.0
pandas sample variance: 10.0


The final deviation must be 4. The same squared-deviation total, 40, gives
variance 8 with divisor 5 and sample variance 10 with divisor 4. NumPy and pandas
agree when the same denominator is requested.


## Why the Correction Works on Average

Expectation, written $E[\cdot]$, is a probability-weighted average over possible
samples. It is different from the value obtained from one particular sample.
For independent observations with common mean $\mu$ and variance $\sigma^2$,
$E[\bar X]=\mu$ and $\mathrm{Var}(\bar X)=\sigma^2/n$.

The squared-deviation identity is

$$\sum_i(X_i-\bar X)^2=\sum_i(X_i-\mu)^2-n(\bar X-\mu)^2.$$

Taking expectations gives

$$E\!\left[\sum_i(X_i-\bar X)^2\right]
=n\sigma^2-n\frac{\sigma^2}{n}=(n-1)\sigma^2.$$

Consequently, $E[s^2]=\sigma^2$. Dividing by $n$ instead gives expected value
$(n-1)\sigma^2/n$, which is too small for that estimation task.
Unbiasedness concerns repeated samples; it does not make every estimate equal
to the population variance. [NumPy variance reference](https://numpy.org/doc/stable/reference/generated/numpy.var.html).


## Inspect Every Sample in a Small Population

Use a population with values 1, 3, and 5, each equally likely. Two independent
draws with replacement produce nine equally likely ordered samples.
`itertools.product()` enumerates them, so this comparison requires no randomness.


In [2]:
from itertools import product

population = np.array([1, 3, 5], dtype=float)
sample_pairs = np.array(list(product(population, repeat=2)))
divisor_n_variances = sample_pairs.var(axis=1, ddof=0)
corrected_variances = sample_pairs.var(axis=1, ddof=1)
print(
    pd.DataFrame(
        {
            "first": sample_pairs[:, 0],
            "second": sample_pairs[:, 1],
            "divisor_n_variance": divisor_n_variances,
            "divisor_n_minus_1_variance": corrected_variances,
        }
    ).to_string(index=False)
)
print(f"Population variance: {population.var(ddof=0):.6f}")
print(f"Average divisor-n variance: {divisor_n_variances.mean():.6f}")
print(f"Average corrected variance: {corrected_variances.mean():.6f}")


 first  second  divisor_n_variance  divisor_n_minus_1_variance
   1.0     1.0                 0.0                         0.0
   1.0     3.0                 1.0                         2.0
   1.0     5.0                 4.0                         8.0
   3.0     1.0                 1.0                         2.0
   3.0     3.0                 0.0                         0.0
   3.0     5.0                 1.0                         2.0
   5.0     1.0                 4.0                         8.0
   5.0     3.0                 1.0                         2.0
   5.0     5.0                 0.0                         0.0
Population variance: 2.666667
Average divisor-n variance: 1.333333
Average corrected variance: 2.666667


The population variance is 8/3, approximately 2.666667. The average
divisor-n estimate is 4/3, while the average corrected estimate is 8/3.
Individual corrected estimates still range from 0 to 8.


## Standard Deviation Still Needs Interpretation

The sample standard deviation is `np.std(sample, ddof=1)`, the square root of
the corrected variance. Taking a square root is nonlinear, so an unbiased
variance estimator does not automatically yield an unbiased standard-deviation
estimator. At least two observations are needed for the `n - 1` divisor.


Specify the denominator and the population being discussed. The n − 1 correction is justified by repeated random sampling, not by equality between every sample and its population.

[Related explanation in the source lecture](https://www.youtube.com/watch?v=9bM65kxOwPA&t=1295s).
